# LeetCode 42 — Trapping Rain Water (Hard) bằng Python

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def trap(self, height: List[int]) -> int:

```

Đây là một bài **Hard** kinh điển, nhưng đừng sợ: ý tưởng cốt lõi chỉ gói trong **một công thức** cho từng cột. Phần khó là làm sao tính công thức đó **nhanh** và **ít bộ nhớ**.

---

## Sau notebook này, bạn sẽ nắm được

1. Hình dung bài toán bằng hình vẽ ASCII: cột ở đâu, nước đọng ở đâu.
2. Công thức vàng: `water[i] = min(maxL[i], maxR[i]) - height[i]`.
3. **Cách 1 — Brute force** `O(n²)`: áp dụng công thức trực tiếp.
4. **Cách 2 — Prefix max / Suffix max** `O(n)` thời gian, `O(n)` bộ nhớ: tính trước "cột cao nhất bên trái / bên phải".
5. **Cách 3 — Monotonic stack** (ngăn xếp đơn điệu): tính nước theo **lớp ngang** thay vì theo cột.
6. **Cách 4 — Two pointers** (hai con trỏ) `O(n)` thời gian, `O(1)` bộ nhớ — cách chính, kèm giải thích kỹ **vì sao được phép quyết định ở bên thấp hơn**.
7. Lỗi hay gặp, mẹo nhận dạng pattern, bài liên quan và 6 bài tập có đáp án.

## 1. Đề bài (theo LeetCode)

Cho `n` số nguyên không âm biểu diễn một **bản đồ độ cao** (elevation map), trong đó mỗi cột có **chiều rộng 1**. Hãy tính xem sau khi mưa, bản đồ này **giữ lại được bao nhiêu đơn vị nước**.

**Example 1**

```text
Input:  height = [0,1,0,2,1,0,1,3,2,1,2,1]
Output: 6
Explanation: The above elevation map (black section) is represented by array
[0,1,0,2,1,0,1,3,2,1,2,1]. In this case, 6 units of rain water (blue section) are being trapped.
```

**Example 2**

```text
Input:  height = [4,2,0,3,2,5]
Output: 9
```

**Constraints**

- `n == height.length`
- `1 <= n <= 2 * 10^4`
- `0 <= height[i] <= 10^5`

Hai điều cần để ý ngay từ đầu:

- Hàm trả về **một số nguyên**: tổng lượng nước, không phải danh sách.
- `n` có thể tới `20 000`, nên lời giải `O(n²)` (khoảng 4 × 10⁸ bước) sẽ quá chậm trong Python.

In [1]:
from typing import List   # LeetCode tự import sẵn; chạy ngoài LeetCode thì ta tự import

height1 = [0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]
height2 = [4, 2, 0, 3, 2, 5]

print("Example 1:", height1, "-> mong đợi 6")
print("Example 2:", height2, "-> mong đợi 9")
print("Số cột của example 1:", len(height1))

Example 1: [0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1] -> mong đợi 6
Example 2: [4, 2, 0, 3, 2, 5] -> mong đợi 9
Số cột của example 1: 12


## 2. Vẽ bài toán ra cho dễ thấy

Với bài này, **vẽ hình** quan trọng hơn mọi công thức. Ta quy ước:

- `#` là một ô của cột (đá/tường),
- `~` là một ô nước,
- dấu cách là không khí,
- trục dưới cùng ghi index cột (chỉ in chữ số hàng đơn vị, nên cột 10, 11 hiện là `0`, `1`).

Example 1 `[0,1,0,2,1,0,1,3,2,1,2,1]` sau khi mưa:

```text
3 |       #    |
2 |   #~~~##~# |
1 | #~##~######|
  +------------+
   012345678901
```

Đếm số dấu `~`: hàng 2 có 4 ô, hàng 1 có 2 ô → tổng **6**. Khớp đáp án!

Cell dưới là một hàm vẽ nhỏ. Bạn **không cần** hiểu hết chi tiết hàm vẽ; chỉ cần biết: truyền `height`, và (tuỳ chọn) danh sách `water` cho biết mỗi cột có bao nhiêu nước.

In [2]:
def draw_elevation(height, water=None):
    """Vẽ cột (#) và nước (~). water[i] = số ô nước nằm trên cột i."""
    n = len(height)
    if water is None:
        water = [0] * n
    top = max(h + w for h, w in zip(height, water))
    for level in range(top, 0, -1):          # vẽ từ trên xuống
        row = ""
        for h, w in zip(height, water):
            if h >= level:
                row += "#"
            elif h + w >= level:
                row += "~"
            else:
                row += " "
        print(f"{level} |{row}|")
    print("  +" + "-" * n + "+")
    print("   " + "".join(str(i % 10) for i in range(n)))


print("Chưa mưa:")
draw_elevation(height1)

Chưa mưa:
3 |       #    |
2 |   #   ## # |
1 | # ## ######|
  +------------+
   012345678901


Bây giờ ta **đếm tay** nước trên từng cột của example 1 (nhìn hình ở trên):

| i | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| height[i] | 0 | 1 | 0 | 2 | 1 | 0 | 1 | 3 | 2 | 1 | 2 | 1 |
| nước trên cột i | 0 | 0 | 1 | 0 | 1 | 2 | 1 | 0 | 0 | 1 | 0 | 0 |

Ta truyền danh sách này vào hàm vẽ để kiểm tra mắt mình có đếm đúng không.

In [3]:
water_by_hand = [0, 0, 1, 0, 1, 2, 1, 0, 0, 1, 0, 0]

print("Sau khi mưa:")
draw_elevation(height1, water_by_hand)
print("Tổng nước đếm tay:", sum(water_by_hand))

Sau khi mưa:
3 |       #    |
2 |   #~~~##~# |
1 | #~##~######|
  +------------+
   012345678901
Tổng nước đếm tay: 6


## 3. Insight cốt lõi: nước trên **một cột** do đâu mà có?

Đừng nghĩ về cả bản đồ cùng lúc. Hãy hỏi một câu nhỏ: **cột `i` chứa được bao nhiêu nước?**

Tưởng tượng bạn đứng trên cột `i` và nhìn sang hai bên:

- `maxL[i]` = cột **cao nhất** từ đầu mảng tới `i` (tính cả chính cột `i`).
- `maxR[i]` = cột **cao nhất** từ `i` tới cuối mảng (tính cả chính cột `i`).

Nước trên cột `i` dâng lên tới **bức tường thấp hơn** trong hai bức tường cao nhất đó (nước tràn qua tường thấp hơn, giống thùng gỗ: thùng chứa được bao nhiêu phụ thuộc vào **thanh gỗ ngắn nhất**). Vậy:

```text
mực nước tại i  = min(maxL[i], maxR[i])
water[i]        = min(maxL[i], maxR[i]) - height[i]
```

**Ví dụ số** — cột `i = 5` của example 1 (`height[5] = 0`):

- Bên trái (index 0..5): `[0,1,0,2,1,0]` → `maxL[5] = 2`.
- Bên phải (index 5..11): `[0,1,3,2,1,2,1]` → `maxR[5] = 3`.
- Mực nước = `min(2, 3) = 2` → `water[5] = 2 - 0 = 2`. Khớp hình vẽ (cột 5 có 2 dấu `~`).

**Vì sao tính cả chính cột `i`?** Vì khi đó `maxL[i] >= height[i]` và `maxR[i] >= height[i]`, nên `water[i]` **không bao giờ âm**. Cột cao hơn mọi thứ hai bên sẽ có `water = 0`, đúng như thực tế.

### Kiến thức Python cần dùng: slicing và `max()` / `min()`

- `a[:k]` lấy các phần tử index `0, 1, ..., k-1` (**không** gồm `k`). Muốn lấy tới hết index `i` thì viết `a[:i+1]`.
- `a[i:]` lấy từ index `i` tới cuối.
- `max(list)` trả về phần tử lớn nhất; `min(x, y)` trả về số nhỏ hơn.

In [4]:
a = [0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]
i = 5

print("a[:i+1] =", a[:i + 1])     # index 0..5
print("a[i:]   =", a[i:])         # index 5..11
print("max(a[:i+1]) =", max(a[:i + 1]))
print("max(a[i:])   =", max(a[i:]))
print("min(2, 3) =", min(2, 3))

a[:i+1] = [0, 1, 0, 2, 1, 0]
a[i:]   = [0, 1, 3, 2, 1, 2, 1]
max(a[:i+1]) = 2
max(a[i:])   = 3
min(2, 3) = 2


In [5]:
def water_at(height, i):
    max_left = max(height[:i + 1])    # cao nhất bên trái, tính cả i
    max_right = max(height[i:])       # cao nhất bên phải, tính cả i
    return min(max_left, max_right) - height[i]

for i in [2, 5, 7, 9, 11]:
    print(f"cột {i:>2}: height = {height1[i]}, water = {water_at(height1, i)}")

cột  2: height = 0, water = 1
cột  5: height = 0, water = 2
cột  7: height = 3, water = 0
cột  9: height = 1, water = 1
cột 11: height = 1, water = 0


## 4. Cách 1 — Brute force: áp công thức cho từng cột

Có công thức cho một cột rồi, cách dễ nghĩ nhất là: **lặp qua mọi cột**, với mỗi cột quét sang trái tìm max, quét sang phải tìm max, rồi cộng `water[i]`.

```text
total = 0
với mỗi i:
    max_left  = max(height[0..i])
    max_right = max(height[i..n-1])
    total += min(max_left, max_right) - height[i]
```

Ta viết hẳn hai vòng `for` bên trong (thay vì dùng slicing) để thấy rõ công việc lặp lại.

In [6]:
def trap_bruteforce(height: List[int]) -> int:
    n = len(height)
    total = 0
    for i in range(n):
        max_left = 0
        for j in range(0, i + 1):          # quét trái: 0..i
            max_left = max(max_left, height[j])
        max_right = 0
        for j in range(i, n):              # quét phải: i..n-1
            max_right = max(max_right, height[j])
        total += min(max_left, max_right) - height[i]
    return total

print(trap_bruteforce(height1))   # 6
print(trap_bruteforce(height2))   # 9

6
9


### Trace cách 1 trên example 1

| i | height | maxL | maxR | mực nước = min | water |
|---:|---:|---:|---:|---:|---:|
| 0 | 0 | 0 | 3 | 0 | 0 |
| 1 | 1 | 1 | 3 | 1 | 0 |
| 2 | 0 | 1 | 3 | 1 | **1** |
| 3 | 2 | 2 | 3 | 2 | 0 |
| 4 | 1 | 2 | 3 | 2 | **1** |
| 5 | 0 | 2 | 3 | 2 | **2** |
| 6 | 1 | 2 | 3 | 2 | **1** |
| 7 | 3 | 3 | 3 | 3 | 0 |
| 8 | 2 | 3 | 2 | 2 | 0 |
| 9 | 1 | 3 | 2 | 2 | **1** |
| 10 | 2 | 3 | 2 | 2 | 0 |
| 11 | 1 | 3 | 1 | 1 | 0 |

Để ý cột 8..11: ở đó `maxR` mới là bức tường thấp hơn, nên mực nước do **bên phải** quyết định. Hàm trace dưới in đúng bảng này.

In [7]:
def trace_bruteforce(height):
    total = 0
    print(" i | h | maxL | maxR | level | water | total")
    for i in range(len(height)):
        max_left = max(height[:i + 1])
        max_right = max(height[i:])
        level = min(max_left, max_right)
        w = level - height[i]
        total += w
        print(f"{i:>2} | {height[i]} | {max_left:>4} | {max_right:>4} | {level:>5} | {w:>5} | {total:>5}")
    return total

_ = trace_bruteforce(height1)   # gán vào _ để Jupyter không in thêm giá trị return

 i | h | maxL | maxR | level | water | total
 0 | 0 |    0 |    3 |     0 |     0 |     0
 1 | 1 |    1 |    3 |     1 |     0 |     0
 2 | 0 |    1 |    3 |     1 |     1 |     1
 3 | 2 |    2 |    3 |     2 |     0 |     1
 4 | 1 |    2 |    3 |     2 |     1 |     2
 5 | 0 |    2 |    3 |     2 |     2 |     4
 6 | 1 |    2 |    3 |     2 |     1 |     5
 7 | 3 |    3 |    3 |     3 |     0 |     5
 8 | 2 |    3 |    2 |     2 |     0 |     5
 9 | 1 |    3 |    2 |     2 |     1 |     6
10 | 2 |    3 |    2 |     2 |     0 |     6
11 | 1 |    3 |    1 |     1 |     0 |     6


### Cách 1 vướng ở đâu?

Với mỗi cột `i`, ta quét `i + 1` phần tử bên trái và `n - i` phần tử bên phải → tổng cộng `n + 1` phần tử mỗi cột, tức khoảng `n × (n + 1)` lần đọc mảng.

- **Time:** `O(n²)` với `n = len(height)`.
- **Space:** `O(1)` (chỉ vài biến).

Với `n = 20 000` thì khoảng **4 × 10⁸** lần đọc → Python chạy hàng phút, bị **TLE** (Time Limit Exceeded).

**Nút thắt:** `maxL[i]` và `maxL[i+1]` gần như giống nhau, vậy mà ta quét lại từ đầu mỗi lần. Công việc bị lặp!

In [8]:
def count_reads_bruteforce(n):
    return sum((i + 1) + (n - i) for i in range(n))

for n in [12, 1_000, 20_000]:
    print(f"n = {n:>6}: số lần đọc mảng ~ {count_reads_bruteforce(n):,}")

n =     12: số lần đọc mảng ~ 156
n =   1000: số lần đọc mảng ~ 1,001,000
n =  20000: số lần đọc mảng ~ 400,020,000


## 5. Cách 2 — Prefix max / Suffix max (tư duy DP)

### Kiến thức nền: "max chạy" (running max / prefix max)

Mấu chốt cải tiến: **max của đoạn `0..i` = max(max của đoạn `0..i-1`, phần tử thứ `i`)**. Không cần quét lại từ đầu, chỉ cần **dùng lại kết quả của bước trước**. Đây chính là tinh thần **DP** (Dynamic Programming — quy hoạch động): lưu đáp án bài nhỏ để dựng đáp án bài lớn hơn.

```text
left_max[0] = a[0]
left_max[i] = max(left_max[i-1], a[i])
```

Ví dụ `a = [3, 1, 4, 1, 5]`:

```text
left_max = [3, 3, 4, 4, 5]
```

Làm ngược từ phải sang trái ta được **suffix max**:

```text
right_max[n-1] = a[n-1]
right_max[i]   = max(right_max[i+1], a[i])
right_max = [5, 5, 5, 5, 5]
```

In [9]:
a = [3, 1, 4, 1, 5]
n = len(a)

left_max = [0] * n          # tạo list n số 0
left_max[0] = a[0]
for i in range(1, n):
    left_max[i] = max(left_max[i - 1], a[i])

right_max = [0] * n
right_max[n - 1] = a[n - 1]
for i in range(n - 2, -1, -1):  # i = n-2, n-3, ..., 0 (bước -1 là đi lùi)
    right_max[i] = max(right_max[i + 1], a[i])

print("a         =", a)
print("left_max  =", left_max)
print("right_max =", right_max)

a         = [3, 1, 4, 1, 5]
left_max  = [3, 3, 4, 4, 5]
right_max = [5, 5, 5, 5, 5]


### Ghép vào bài toán

1. Một lượt trái → phải: dựng `left_max`.
2. Một lượt phải → trái: dựng `right_max`.
3. Một lượt nữa: cộng `min(left_max[i], right_max[i]) - height[i]`.

Ba lượt, mỗi lượt `O(n)` → tổng `O(n)`. Công thức **không đổi** so với cách 1; ta chỉ đổi cách **lấy** `maxL`, `maxR` từ "quét lại" sang "tra bảng".

In [10]:
def trap_prefix_suffix(height: List[int]) -> int:
    n = len(height)
    left_max = [0] * n
    right_max = [0] * n

    left_max[0] = height[0]
    for i in range(1, n):
        left_max[i] = max(left_max[i - 1], height[i])

    right_max[n - 1] = height[n - 1]
    for i in range(n - 2, -1, -1):
        right_max[i] = max(right_max[i + 1], height[i])

    total = 0
    for i in range(n):
        total += min(left_max[i], right_max[i]) - height[i]
    return total

print(trap_prefix_suffix(height1))   # 6
print(trap_prefix_suffix(height2))   # 9

6
9


In [11]:
def trace_prefix_suffix(height):
    n = len(height)
    left_max, right_max = [0] * n, [0] * n
    left_max[0] = height[0]
    for i in range(1, n):
        left_max[i] = max(left_max[i - 1], height[i])
    right_max[n - 1] = height[n - 1]
    for i in range(n - 2, -1, -1):
        right_max[i] = max(right_max[i + 1], height[i])
    water = [min(left_max[i], right_max[i]) - height[i] for i in range(n)]
    print("height    =", height)
    print("left_max  =", left_max)
    print("right_max =", right_max)
    print("water     =", water)
    print("tổng      =", sum(water))
    return water

water1 = trace_prefix_suffix(height1)
assert water1 == water_by_hand   # khớp với bảng đếm tay ở mục 2

height    = [0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]
left_max  = [0, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 3]
right_max = [3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 1]
water     = [0, 0, 1, 0, 1, 2, 1, 0, 0, 1, 0, 0]
tổng      = 6


### Độ phức tạp cách 2

- **Time:** `O(n)` — ba vòng lặp rời nhau, mỗi vòng `n` bước.
- **Space:** `O(n)` — hai mảng phụ `left_max`, `right_max`.

**Còn vướng gì?** Nhìn kỹ công thức `min(left_max[i], right_max[i])`: ta chỉ cần **cái nhỏ hơn** trong hai số. Liệu có thể biết "cái nhỏ hơn" mà **không cần lưu cả hai mảng**? Câu hỏi này dẫn tới cách 4 (two pointers). Trước đó, ta học một góc nhìn hoàn toàn khác: tính nước theo **lớp ngang**.

## 6. Cách 3 — Monotonic stack: đổ nước theo lớp ngang

### Kiến thức nền: stack (ngăn xếp)

**Stack** giống một chồng đĩa: đĩa đặt vào **sau cùng** sẽ được lấy ra **đầu tiên** — gọi là **LIFO** (Last In, First Out).

Trong Python dùng luôn `list`:

| Thao tác | Code | Ý nghĩa |
|---|---|---|
| push (đặt vào đỉnh) | `stack.append(x)` | thêm vào cuối list |
| pop (lấy ra từ đỉnh) | `stack.pop()` | bỏ và trả về phần tử cuối |
| peek (nhìn đỉnh) | `stack[-1]` | xem phần tử cuối, không bỏ |
| rỗng chưa? | `if not stack:` | list rỗng thì `not stack` là `True` |

In [12]:
stack = []
stack.append(10)
stack.append(20)
stack.append(30)
print("stack sau 3 lần push:", stack)
print("đỉnh stack[-1]     :", stack[-1])
print("pop ra             :", stack.pop())
print("stack còn lại      :", stack)
print("stack rỗng?        :", not stack)

stack sau 3 lần push: [10, 20, 30]
đỉnh stack[-1]     : 30
pop ra             : 30
stack còn lại      : [10, 20]
stack rỗng?        : False


### Kiến thức nền: monotonic stack (ngăn xếp đơn điệu)

**Monotonic decreasing stack** là stack mà chiều cao (theo index lưu bên trong) **giảm dần từ đáy lên đỉnh**. Quy tắc giữ tính "giảm dần":

> Trước khi push phần tử mới `x`, **pop hết** các phần tử ở đỉnh **thấp hơn** `x`.

Mỗi lần pop là một khoảnh khắc đặc biệt: phần tử bị pop vừa **tìm thấy cột cao hơn nó ở bên phải** (chính là `x`), và cột ngay dưới nó trong stack là **cột cao hơn (hoặc bằng) nó ở bên trái**. Có tường hai bên = có chỗ đọng nước!

Ví dụ mini với `a = [3, 1, 2]` (stack lưu **index**):

In [13]:
a = [3, 1, 2]
stack = []   # lưu index, chiều cao giảm dần từ đáy lên đỉnh
for i, x in enumerate(a):    # enumerate cho cả index i lẫn giá trị x
    while stack and a[stack[-1]] < x:
        top = stack.pop()
        print(f"i={i} (h={x}): pop index {top} (h={a[top]}) vì thấp hơn {x}")
    stack.append(i)
    print(f"i={i}: push {i} -> stack = {stack}, chiều cao = {[a[j] for j in stack]}")

i=0: push 0 -> stack = [0], chiều cao = [3]
i=1: push 1 -> stack = [0, 1], chiều cao = [3, 1]
i=2 (h=2): pop index 1 (h=1) vì thấp hơn 2
i=2: push 2 -> stack = [0, 2], chiều cao = [3, 2]


### Áp dụng: mỗi lần pop là đổ một "lớp nước" hình chữ nhật

Duyệt `i` từ trái sang phải. Khi `height[i] > height[stack[-1]]`:

1. `bottom = stack.pop()` — cột "đáy" của vũng nước.
2. Nếu stack rỗng → không có tường trái → **dừng** (`break`), nước tràn ra ngoài.
3. `left = stack[-1]` — tường trái (cao hơn hoặc bằng đáy).
4. Tường phải là `i`. Lớp nước nằm **giữa** hai tường:
   - `width  = i - left - 1` (số cột nằm giữa, **không** tính hai tường),
   - `bounded_height = min(height[left], height[i]) - height[bottom]` (độ dày lớp nước),
   - `total += width * bounded_height`.

Khác biệt với cách 1, 2: ở đây ta **không** tính nước theo từng cột đứng, mà theo từng **dải ngang** nằm trên cột đáy. Tổng diện tích vẫn như nhau.

In [14]:
def trap_monotonic_stack(height: List[int]) -> int:
    stack = []          # index các cột, chiều cao giảm dần từ đáy lên đỉnh
    total = 0
    for i, h in enumerate(height):
        while stack and h > height[stack[-1]]:
            bottom = stack.pop()
            if not stack:               # không có tường trái -> nước tràn
                break
            left = stack[-1]
            width = i - left - 1
            bounded_height = min(height[left], h) - height[bottom]
            total += width * bounded_height
        stack.append(i)
    return total

print(trap_monotonic_stack(height1))   # 6
print(trap_monotonic_stack(height2))   # 9

6
9


### Trace cách 3 trên example 2 `[4,2,0,3,2,5]`

| i | height[i] | pop bottom | left (tường trái) | width | bounded_height | + nước | total | stack sau bước |
|---:|---:|---|---|---:|---:|---:|---:|---|
| 0 | 4 | — | — | | | | 0 | `[0]` |
| 1 | 2 | — | — | | | | 0 | `[0,1]` |
| 2 | 0 | — | — | | | | 0 | `[0,1,2]` |
| 3 | 3 | 2 (h=0) | 1 (h=2) | 1 | min(2,3)-0 = 2 | 2 | 2 | |
| 3 | 3 | 1 (h=2) | 0 (h=4) | 2 | min(4,3)-2 = 1 | 2 | 4 | `[0,3]` |
| 4 | 2 | — | — | | | | 4 | `[0,3,4]` |
| 5 | 5 | 4 (h=2) | 3 (h=3) | 1 | min(3,5)-2 = 1 | 1 | 5 | |
| 5 | 5 | 3 (h=3) | 0 (h=4) | 4 | min(4,5)-3 = 1 | 4 | 9 | |
| 5 | 5 | 0 (h=4) | stack rỗng → `break` | | | | 9 | `[5]` |

Nhánh đáng chú ý: ở `i = 5`, sau khi pop index `0` thì stack rỗng — không còn tường trái, nên **không** cộng gì và `break`.

In [15]:
def trace_monotonic_stack(height):
    stack, total, layers = [], 0, []
    for i, h in enumerate(height):
        while stack and h > height[stack[-1]]:
            bottom = stack.pop()
            if not stack:
                print(f"i={i}: pop {bottom} -> stack rỗng, không có tường trái -> break")
                break
            left = stack[-1]
            width = i - left - 1
            bh = min(height[left], h) - height[bottom]
            total += width * bh
            layers.append((left, i, height[bottom], height[bottom] + bh))
            print(f"i={i}: pop bottom={bottom}(h={height[bottom]}), left={left}(h={height[left]}), "
                  f"width={width}, bh={bh}, +{width * bh} -> total={total}")
        stack.append(i)
        print(f"i={i}: push -> stack={stack}")
    return total, layers

total2, layers2 = trace_monotonic_stack(height2)
print("Tổng:", total2)

i=0: push -> stack=[0]
i=1: push -> stack=[0, 1]
i=2: push -> stack=[0, 1, 2]
i=3: pop bottom=2(h=0), left=1(h=2), width=1, bh=2, +2 -> total=2
i=3: pop bottom=1(h=2), left=0(h=4), width=2, bh=1, +2 -> total=4
i=3: push -> stack=[0, 3]
i=4: push -> stack=[0, 3, 4]
i=5: pop bottom=4(h=2), left=3(h=3), width=1, bh=1, +1 -> total=5
i=5: pop bottom=3(h=3), left=0(h=4), width=4, bh=1, +4 -> total=9
i=5: pop 0 -> stack rỗng, không có tường trái -> break
i=5: push -> stack=[5]
Tổng: 9


### Nhìn thấy các "lớp ngang"

Mỗi lần cộng nước ở trên tạo ra một hình chữ nhật. Ta đánh nhãn `a`, `b`, `c`, `d` theo **thứ tự** được đổ vào (vẽ bằng dữ liệu `layers2` mà hàm trace vừa trả về):

- `a`: giữa cột 1 và 3, phủ cột 2, dày 2 → 2 ô.
- `b`: giữa cột 0 và 3, phủ cột 1..2, dày 1 → 2 ô.
- `c`: giữa cột 3 và 5, phủ cột 4, dày 1 → 1 ô.
- `d`: giữa cột 0 và 5, phủ cột 1..4, dày 1 → 4 ô.

Tổng `2 + 2 + 1 + 4 = 9`.

In [16]:
def draw_layers(height, layers):
    top = max(height)
    grid = [[" "] * len(height) for _ in range(top + 1)]
    for c, h in enumerate(height):
        for level in range(1, h + 1):
            grid[level][c] = "#"
    for k, (left, right, low, high) in enumerate(layers):
        label = "abcdefghijklmnopqrstuvwxyz"[k % 26]
        for c in range(left + 1, right):            # các cột nằm giữa hai tường
            for level in range(low + 1, high + 1):  # các hàng của lớp nước
                grid[level][c] = label
    for level in range(top, 0, -1):
        print(f"{level} |{''.join(grid[level])}|")
    print("  +" + "-" * len(height) + "+")
    print("   " + "".join(str(i % 10) for i in range(len(height))))

draw_layers(height2, layers2)

5 |     #|
4 |#dddd#|
3 |#bb#c#|
2 |##a###|
1 |##a###|
  +------+
   012345


### Độ phức tạp cách 3

- **Time:** `O(n)`. Vòng `while` lồng trong `for` trông như `O(n²)`, nhưng **mỗi index được push đúng 1 lần và pop tối đa 1 lần**, nên tổng số lần pop ≤ `n`. Đây là phân tích **amortized** (khấu hao): tính tổng công việc của cả quá trình thay vì từng bước.
- **Space:** `O(n)` cho stack (ví dụ mảng giảm dần `[5,4,3,2,1]` sẽ push hết mà không pop).

Cách này đáng học vì kỹ thuật monotonic stack xuất hiện ở rất nhiều bài khác (Largest Rectangle in Histogram, Daily Temperatures...). Nhưng bộ nhớ vẫn `O(n)`. Giờ ta tới cách tối ưu nhất.

## 7. Cách 4 — Two pointers: `O(n)` thời gian, `O(1)` bộ nhớ (cách chính)

### Kiến thức nền: two pointers (hai con trỏ)

**Two pointers** là kỹ thuật dùng hai biến index, thường là `l` (left) bắt đầu ở đầu mảng và `r` (right) bắt đầu ở cuối mảng. Mỗi bước, ta **chọn một** con trỏ để dịch vào trong (`l += 1` hoặc `r -= 1`), cho tới khi hai con trỏ gặp nhau. Mỗi bước loại được một phần tử, nên tổng cộng chỉ `n - 1` bước.

Câu hỏi quan trọng nhất của mọi bài two pointers: **"Mỗi bước, dựa vào đâu để quyết định dịch con trỏ nào?"**

Ví dụ mini: duyệt mảng từ hai đầu, mỗi bước dịch con trỏ đang đứng ở **số nhỏ hơn**.

In [17]:
a = [2, 7, 1, 8, 3]
l, r = 0, len(a) - 1
step = 1
while l < r:
    if a[l] < a[r]:
        print(f"bước {step}: l={l} (a={a[l]}), r={r} (a={a[r]}) -> bên trái nhỏ hơn, l += 1")
        l += 1
    else:
        print(f"bước {step}: l={l} (a={a[l]}), r={r} (a={a[r]}) -> bên phải nhỏ hơn/bằng, r -= 1")
        r -= 1
    step += 1
print(f"dừng khi l == r == {l}")

bước 1: l=0 (a=2), r=4 (a=3) -> bên trái nhỏ hơn, l += 1
bước 2: l=1 (a=7), r=4 (a=3) -> bên phải nhỏ hơn/bằng, r -= 1
bước 3: l=1 (a=7), r=3 (a=8) -> bên trái nhỏ hơn, l += 1
bước 4: l=2 (a=1), r=3 (a=8) -> bên trái nhỏ hơn, l += 1
dừng khi l == r == 3


### Mấu chốt: không cần biết **chính xác** cả hai max, chỉ cần biết **cái nào nhỏ hơn**

Nhắc lại công thức: `water[i] = min(maxL[i], maxR[i]) - height[i]`.

Ta giữ hai biến:

- `left_max`  = max của `height[0..l]` — mọi thứ con trỏ trái **đã đi qua**,
- `right_max` = max của `height[r..n-1]` — mọi thứ con trỏ phải **đã đi qua**.

Phần ở giữa `(l, r)` **chưa được nhìn thấy**. Hình dung:

```text
index:   0 ........ l   [ vùng chưa xem ]   r ........ n-1
         \___________/                      \___________/
           left_max                           right_max
```

**Trường hợp `left_max < right_max`** — xét cột `l`:

- `maxL[l]` thật **chính là** `left_max` (ta đã thấy toàn bộ bên trái của `l`).
- `maxR[l]` thật = max của `height[l..n-1]`. Đoạn này **chứa** đoạn `r..n-1`, nên `maxR[l] >= right_max > left_max`.
- Vậy `min(maxL[l], maxR[l]) = left_max`. **Chắc chắn**, dù vùng giữa có cột cao bao nhiêu đi nữa!
- → `water[l] = left_max - height[l]`, rồi `l += 1`.

**Trường hợp `left_max >= right_max`** — đối xứng, xét cột `r`:

- `maxR[r]` thật chính là `right_max`; `maxL[r] >= left_max >= right_max`.
- Vậy `min = right_max` → `water[r] = right_max - height[r]`, rồi `r -= 1`.

> **Câu chốt:** bên nào có max **thấp hơn** thì bên đó là "thanh gỗ ngắn" — mực nước ở cột đó đã bị chặn bởi chính max của nó. Vùng chưa xem chỉ có thể làm bức tường bên kia **cao thêm**, không bao giờ làm nó thấp đi. Vì thế ta được phép **quyết định ngay** cột ở phía thấp hơn.

Ngược lại, ta **không** được quyết định ở phía cao hơn: với cột ở phía cao, bức tường "ngắn" có thể nằm trong vùng chưa xem, nên chưa biết mực nước.

### Thuật toán

```text
l, r = 0, n - 1
left_max = right_max = 0
total = 0
while l < r:
    left_max  = max(left_max, height[l])     # cập nhật trước, nên water không âm
    right_max = max(right_max, height[r])
    if left_max < right_max:
        total += left_max - height[l];  l += 1
    else:
        total += right_max - height[r]; r -= 1
```

**Khi bằng nhau** (`left_max == right_max`) thì xử lý bên nào cũng đúng (cả hai lập luận đều cho `min = right_max = left_max`); ta chọn bên phải cho gọn code.

**Cột cuối cùng** khi `l == r` không cần xử lý: đó luôn là một cột cao nhất của mảng — con trỏ đã đứng trên cột cao nhất chỉ bị dịch khi con trỏ kia cũng đã gặp một cột cao bằng nó. Cột cao nhất thì không chứa nước. Dòng cuối của trace bên dưới cho thấy điều này.

In [18]:
def trap_two_pointers(height: List[int]) -> int:
    l, r = 0, len(height) - 1
    left_max = right_max = 0
    total = 0
    while l < r:
        left_max = max(left_max, height[l])
        right_max = max(right_max, height[r])
        if left_max < right_max:
            total += left_max - height[l]     # cột l: mực nước = left_max
            l += 1
        else:
            total += right_max - height[r]    # cột r: mực nước = right_max
            r -= 1
    return total

print(trap_two_pointers(height1))   # 6
print(trap_two_pointers(height2))   # 9

6
9


### Trace cách 4 trên example 1 `[0,1,0,2,1,0,1,3,2,1,2,1]`

(`left_max`, `right_max` là giá trị **sau khi cập nhật** ở đầu vòng.)

| bước | l | r | h[l] | h[r] | left_max | right_max | so sánh | xử lý cột | + nước | total |
|---:|---:|---:|---:|---:|---:|---:|---|---|---:|---:|
| 1 | 0 | 11 | 0 | 1 | 0 | 1 | 0 < 1 | 0 (trái) | 0 | 0 |
| 2 | 1 | 11 | 1 | 1 | 1 | 1 | bằng | 11 (phải) | 0 | 0 |
| 3 | 1 | 10 | 1 | 2 | 1 | 2 | 1 < 2 | 1 (trái) | 0 | 0 |
| 4 | 2 | 10 | 0 | 2 | 1 | 2 | 1 < 2 | 2 (trái) | **1** | 1 |
| 5 | 3 | 10 | 2 | 2 | 2 | 2 | bằng | 10 (phải) | 0 | 1 |
| 6 | 3 | 9 | 2 | 1 | 2 | 2 | bằng | 9 (phải) | **1** | 2 |
| 7 | 3 | 8 | 2 | 2 | 2 | 2 | bằng | 8 (phải) | 0 | 2 |
| 8 | 3 | 7 | 2 | 3 | 2 | 3 | 2 < 3 | 3 (trái) | 0 | 2 |
| 9 | 4 | 7 | 1 | 3 | 2 | 3 | 2 < 3 | 4 (trái) | **1** | 3 |
| 10 | 5 | 7 | 0 | 3 | 2 | 3 | 2 < 3 | 5 (trái) | **2** | 5 |
| 11 | 6 | 7 | 1 | 3 | 2 | 3 | 2 < 3 | 6 (trái) | **1** | 6 |

Dừng khi `l = r = 7` (cột cao nhất, `h = 3`).

**Bước 6 rất đáng ngẫm:** ta tính nước cho cột 9 bằng `right_max = 2` dù **chưa nhìn thấy** cột 7 (cao 3) ở vùng giữa. Thật ra `maxL[9] = 3`, nhưng điều đó không quan trọng: ta chỉ cần biết `maxL[9] >= 2`, nên `min(maxL[9], maxR[9]) = 2`. **Biết đủ, không cần biết hết.**

In [19]:
def trace_two_pointers(height):
    l, r = 0, len(height) - 1
    left_max = right_max = 0
    total, step = 0, 1
    print("bước |  l |  r | lmax | rmax | xử lý      | +  | total")
    while l < r:
        left_max = max(left_max, height[l])
        right_max = max(right_max, height[r])
        l0, r0 = l, r                     # nhớ vị trí trước khi dịch
        if left_max < right_max:
            add, col, side = left_max - height[l], l, "trái"
            l += 1
        else:
            add, col, side = right_max - height[r], r, "phải"
            r -= 1
        total += add
        print(f"{step:>4} | {l0:>2} | {r0:>2} | {left_max:>4} | {right_max:>4} |"
              f" cột {col:>2} {side:<4} | {add:>2} | {total:>5}")
        step += 1
    print(f"dừng: l = r = {l}, height[{l}] = {height[l]} (max của mảng = {max(height)})")
    return total

_ = trace_two_pointers(height1)

bước |  l |  r | lmax | rmax | xử lý      | +  | total
   1 |  0 | 11 |    0 |    1 | cột  0 trái |  0 |     0
   2 |  1 | 11 |    1 |    1 | cột 11 phải |  0 |     0
   3 |  1 | 10 |    1 |    2 | cột  1 trái |  0 |     0
   4 |  2 | 10 |    1 |    2 | cột  2 trái |  1 |     1
   5 |  3 | 10 |    2 |    2 | cột 10 phải |  0 |     1
   6 |  3 |  9 |    2 |    2 | cột  9 phải |  1 |     2
   7 |  3 |  8 |    2 |    2 | cột  8 phải |  0 |     2
   8 |  3 |  7 |    2 |    3 | cột  3 trái |  0 |     2
   9 |  4 |  7 |    2 |    3 | cột  4 trái |  1 |     3
  10 |  5 |  7 |    2 |    3 | cột  5 trái |  2 |     5
  11 |  6 |  7 |    2 |    3 | cột  6 trái |  1 |     6
dừng: l = r = 7, height[7] = 3 (max của mảng = 3)


### Vì sao đúng? (invariant bằng lời)

Ở **đầu mỗi vòng lặp** (sau khi cập nhật hai max), luôn đúng:

1. `left_max = max(height[0..l])` và `right_max = max(height[r..n-1])`.
2. `total` = tổng nước **chính xác** của mọi cột đã bị loại (các cột `< l` và `> r`).

Mỗi vòng ta loại đúng một cột ở phía có max nhỏ hơn, và theo lập luận ở trên, nước cộng thêm cho cột đó **bằng đúng** `water[i]` thật. Khi `l == r`, mọi cột trừ một cột cao nhất đã được loại, mà cột cao nhất có nước 0 → `total` là đáp án.

Cell dưới **kiểm chứng bằng máy**: ghi lại nước mà two pointers gán cho từng cột, rồi so với `water1` (từ cách 2, tính theo công thức đầy đủ).

In [20]:
def water_per_column_two_pointers(height):
    n = len(height)
    water = [0] * n
    l, r = 0, n - 1
    left_max = right_max = 0
    while l < r:
        left_max = max(left_max, height[l])
        right_max = max(right_max, height[r])
        if left_max < right_max:
            water[l] = left_max - height[l]
            l += 1
        else:
            water[r] = right_max - height[r]
            r -= 1
    return water

print("two pointers:", water_per_column_two_pointers(height1))
print("cách 2      :", water1)
assert water_per_column_two_pointers(height1) == water1
print("Từng cột khớp nhau!")

two pointers: [0, 0, 1, 0, 1, 2, 1, 0, 0, 1, 0, 0]
cách 2      : [0, 0, 1, 0, 1, 2, 1, 0, 0, 1, 0, 0]
Từng cột khớp nhau!


### Độ phức tạp cách 4

Gọi `n = len(height)`.

- **Time:** `O(n)` — mỗi vòng dịch một con trỏ, tổng `n - 1` vòng.
- **Space:** `O(1)` — chỉ dùng `l, r, left_max, right_max, total`.

Đây là lời giải tốt nhất về cả thời gian lẫn bộ nhớ, và cũng là cách hay được hỏi khi phỏng vấn.

## 8. Code nộp LeetCode

Bản submit gọn, dùng two pointers. Code submit không `print`, đúng signature `trap(self, height: List[int]) -> int`.

In [21]:
class Solution:
    def trap(self, height: List[int]) -> int:
        l, r = 0, len(height) - 1
        left_max = right_max = 0
        total = 0
        while l < r:
            left_max = max(left_max, height[l])
            right_max = max(right_max, height[r])
            if left_max < right_max:
                total += left_max - height[l]
                l += 1
            else:
                total += right_max - height[r]
                r -= 1
        return total


sol = Solution()
tests = [
    ([0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1], 6),   # example 1
    ([4, 2, 0, 3, 2, 5], 9),                     # example 2
    ([5], 0),                                    # n = 1 (biên nhỏ nhất)
    ([2, 0], 0),                                 # 2 cột: không có chỗ giữa
    ([3, 0, 3], 3),                              # một hố đơn giản
    ([1, 2, 3, 4, 5], 0),                        # tăng dần: nước tràn trái
    ([5, 4, 3, 2, 1], 0),                        # giảm dần: nước tràn phải
    ([0, 0, 0], 0),                              # toàn số 0
    ([4, 4, 4, 4], 0),                           # toàn bằng nhau
    ([5, 0, 0, 0, 1], 3),                        # tường thấp quyết định
    ([100000, 0, 100000], 100000),               # giá trị lớn nhất theo constraints
]
for h, expected in tests:
    got = sol.trap(h)
    assert got == expected, (h, got, expected)
    print(f"{str(h):<40} -> {got:>6} PASS")

[0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]     ->      6 PASS
[4, 2, 0, 3, 2, 5]                       ->      9 PASS
[5]                                      ->      0 PASS
[2, 0]                                   ->      0 PASS
[3, 0, 3]                                ->      3 PASS
[1, 2, 3, 4, 5]                          ->      0 PASS
[5, 4, 3, 2, 1]                          ->      0 PASS
[0, 0, 0]                                ->      0 PASS
[4, 4, 4, 4]                             ->      0 PASS
[5, 0, 0, 0, 1]                          ->      3 PASS
[100000, 0, 100000]                      -> 100000 PASS


### Kiểm thử đối chiếu chéo 4 cách

Brute force chậm nhưng **dễ tin là đúng** vì bám sát công thức. Ta sinh ngẫu nhiên nhiều mảng nhỏ (cố định `random.seed` để lần nào chạy cũng ra cùng bộ test) và so 4 cách với nhau.

In [22]:
import random

random.seed(42)
for _ in range(2000):
    n = random.randint(1, 12)
    h = [random.randint(0, 6) for _ in range(n)]
    expected = trap_bruteforce(h)
    assert trap_prefix_suffix(h) == expected, h
    assert trap_monotonic_stack(h) == expected, h
    assert trap_two_pointers(h) == expected, h
    assert Solution().trap(h) == expected, h
print("2000 mảng ngẫu nhiên: cả 4 cách cho cùng kết quả!")

# Một mảng lớn đúng cỡ constraints (n = 20 000): chỉ chạy các cách O(n)
big = [random.randint(0, 10**5) for _ in range(20_000)]
print(trap_prefix_suffix(big) == trap_monotonic_stack(big) == trap_two_pointers(big))

2000 mảng ngẫu nhiên: cả 4 cách cho cùng kết quả!


True


## 9. So sánh các cách

`n = len(height)`.

| Cách | Ý tưởng | Time | Space | Nhận xét |
|---|---|---:|---:|---|
| 1. Brute force | Mỗi cột quét trái/phải tìm max | `O(n²)` | `O(1)` | Dễ hiểu, TLE với `n = 2·10⁴` |
| 2. Prefix/Suffix max | Tính trước `left_max[]`, `right_max[]` | `O(n)` | `O(n)` | Cầu nối tự nhiên từ công thức |
| 3. Monotonic stack | Đổ nước theo lớp ngang khi gặp cột cao hơn | `O(n)` | `O(n)` | Dạy kỹ thuật stack, góc nhìn "lớp" |
| 4. Two pointers | Quyết định cột ở phía có max thấp hơn | `O(n)` | `O(1)` | **Tối ưu nhất**, nên nộp |

Lộ trình suy nghĩ: **công thức cho 1 cột** → bỏ quét lặp bằng **mảng max chạy** → nhận ra chỉ cần **cái nhỏ hơn** của hai max → **hai con trỏ**.

## 10. Lỗi người mới hay gặp

### Lỗi 1: Nghĩ mực nước = cột cao nhất của cả mảng

Sai: `water[i] = max(height) - height[i]`. Nước chỉ dâng tới **tường thấp hơn** trong hai bên, không phải tường cao nhất toàn cục.

### Lỗi 2: Two pointers dịch nhầm phía **cao hơn**

Sai: `if left_max > right_max: xử lý cột l`. Ở phía cao, tường "ngắn" có thể nằm trong vùng chưa xem, nên ta chưa biết mực nước → kết quả sai.

### Lỗi 3: Monotonic stack tính sai `width`

Sai: `width = i - left`. Hai tường `left` và `i` **không** chứa nước; số cột ở giữa là `i - left - 1`.

### Lỗi 4: Monotonic stack quên kiểm tra stack rỗng sau khi pop

Sai: pop xong lấy luôn `stack[-1]`. Nếu stack rỗng → `IndexError`. Phải `if not stack: break`.

### Lỗi 5: Không tính chính cột `i` vào max

Sai: `max_left = max(height[:i])`. Với `i = 0` thì `max([])` gây `ValueError`; và nếu cột `i` cao hơn mọi cột bên trái, `water[i]` sẽ **âm**. Dùng `height[:i+1]` (hoặc cập nhật max **trước** khi cộng nước như trong cách 4).

Cell dưới chạy thử các bản sai để thấy tận mắt.

In [23]:
def wrong_global_max(height):                 # Lỗi 1
    return sum(max(height) - x for x in height)

def wrong_two_pointers_taller_side(height):   # Lỗi 2
    l, r = 0, len(height) - 1
    left_max = right_max = 0
    total = 0
    while l < r:
        left_max = max(left_max, height[l])
        right_max = max(right_max, height[r])
        if left_max > right_max:              # SAI: xử lý phía cao hơn
            total += left_max - height[l]; l += 1
        else:
            total += right_max - height[r]; r -= 1
    return total

def wrong_stack_width(height):                # Lỗi 3
    stack, total = [], 0
    for i, h in enumerate(height):
        while stack and h > height[stack[-1]]:
            bottom = stack.pop()
            if not stack:
                break
            left = stack[-1]
            total += (i - left) * (min(height[left], h) - height[bottom])   # SAI: thiếu -1
        stack.append(i)
    return total

def wrong_no_empty_check(height):             # Lỗi 4
    stack, total = [], 0
    for i, h in enumerate(height):
        while stack and h > height[stack[-1]]:
            bottom = stack.pop()
            left = stack[-1]                  # SAI: stack có thể rỗng
            total += (i - left - 1) * (min(height[left], h) - height[bottom])
        stack.append(i)
    return total

print("đúng                  :", trap_two_pointers(height1))
print("lỗi 1 (max toàn cục)  :", wrong_global_max(height1))
print("lỗi 2 (dịch phía cao) :", wrong_two_pointers_taller_side(height1))
print("lỗi 3 (width thiếu -1):", wrong_stack_width(height1))
try:
    wrong_no_empty_check(height1)
except IndexError as e:
    print("lỗi 4 (không check rỗng): IndexError -", e)

đúng                  : 6
lỗi 1 (max toàn cục)  : 22
lỗi 2 (dịch phía cao) : 14
lỗi 3 (width thiếu -1): 10
lỗi 4 (không check rỗng): IndexError - list index out of range


## 11. Mẹo nhận dạng pattern

- Đề có **mảng chiều cao / histogram** và hỏi về **nước, diện tích, khoảng giữa hai bức tường** → nghĩ ngay tới "cột cao nhất bên trái / bên phải".
- Đáp án ở mỗi vị trí phụ thuộc vào **max/min của toàn bộ phía trái và phía phải** → **prefix/suffix array** (giống bài Product of Array Except Self).
- Câu hỏi kiểu "phần tử **lớn hơn gần nhất** bên trái/phải" → **monotonic stack**.
- Công thức có dạng `min(A, B)` và **A, B tăng dần khi ta mở rộng từ hai đầu** → thử **two pointers**: luôn xử lý phía có giá trị nhỏ hơn, vì phía đó đã bị "khóa" kết quả.
- Muốn từ `O(n)` bộ nhớ xuống `O(1)`: tự hỏi "mình có thật sự cần **cả hai** mảng phụ không, hay chỉ cần một so sánh?".

## 12. Biến thể và bài liên quan

| Bài | Liên hệ |
|---|---|
| LeetCode 11 — Container With Most Water | Two pointers từ hai đầu, dịch phía **thấp hơn** (nhưng chỉ chọn 2 đường, không có cột ở giữa chắn nước) |
| LeetCode 407 — Trapping Rain Water II | Phiên bản 2D: dùng heap (min-heap) mở rộng từ biên vào trong, cùng ý tưởng "tường thấp nhất quyết định" |
| LeetCode 84 — Largest Rectangle in Histogram | Monotonic stack trên histogram |
| LeetCode 739 — Daily Temperatures | Monotonic stack tìm "phần tử lớn hơn gần nhất bên phải" |
| LeetCode 238 — Product of Array Except Self | Prefix/suffix array |

## 13. Bài tập vận dụng

Làm theo thứ tự, từ dễ tới khó. Gợi ý ở mục 14, đáp án ở **cuối notebook**.

### Bài 1 — Tính tay (không chạy code)

Với mỗi mảng, hãy lập bảng `maxL`, `maxR`, `water` rồi tính tổng nước:

- (a) `[3, 0, 2, 0, 4]`
- (b) `[2, 0, 2]`
- (c) `[5, 4, 3, 2, 1]`

Sau đó (tuỳ chọn) dùng `draw_elevation` để vẽ và kiểm tra lại.

### Bài 2 — Đoán đường đi của hai con trỏ

Chạy tay `trap_two_pointers([4, 2, 0, 3, 2, 5])`. Ở mỗi bước, con trỏ nào được dịch? Có bước nào dịch con trỏ phải không? Vì sao? Tổng nước bằng bao nhiêu?

### Bài 3 — Điền TODO: cách prefix/suffix max

Hoàn thiện hàm theo 3 bước: dựng `left_max`, dựng `right_max`, cộng nước.

In [24]:
def exercise_trap_prefix_suffix(height: List[int]) -> int:
    n = len(height)
    left_max = [0] * n
    right_max = [0] * n
    # TODO 1: left_max[0] = ..., rồi vòng for i từ 1 tới n-1
    # TODO 2: right_max[n-1] = ..., rồi vòng for i từ n-2 lùi về 0
    # TODO 3: cộng min(left_max[i], right_max[i]) - height[i] cho mọi i
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_trap_prefix_suffix([0,1,0,2,1,0,1,3,2,1,2,1]) == 6
# assert exercise_trap_prefix_suffix([4,2,0,3,2,5]) == 9
# assert exercise_trap_prefix_suffix([7]) == 0

### Bài 4 — Tự viết lại two pointers theo phiên bản "so sánh `height[l]` với `height[r]`"

Một phiên bản phổ biến khác **so sánh chiều cao hai con trỏ** thay vì so sánh hai max:

- Nếu `height[l] < height[r]`: xử lý cột `l` (nếu `height[l] >= left_max` thì cập nhật `left_max`, ngược lại cộng `left_max - height[l]`), rồi `l += 1`.
- Ngược lại: xử lý cột `r` tương tự với `right_max`, rồi `r -= 1`.

Viết lại hàm này và tự giải thích bằng lời: khi `height[l] < height[r]`, vì sao chắc chắn bên trái là "thanh gỗ ngắn" (tức `left_max < right_max`)? (Gợi ý ở mục 14.)

In [25]:
def exercise_trap_two_pointers_v2(height: List[int]) -> int:
    l, r = 0, len(height) - 1
    left_max = right_max = 0
    total = 0
    # TODO: vòng while l < r, so sánh height[l] với height[r]
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_trap_two_pointers_v2([0,1,0,2,1,0,1,3,2,1,2,1]) == 6
# assert exercise_trap_two_pointers_v2([4,2,0,3,2,5]) == 9
# assert exercise_trap_two_pointers_v2([5,0,0,0,1]) == 3

### Bài 5 — Biến thể: trả về lượng nước trên **từng cột**

Viết hàm trả về list `water` (độ dài `n`) thay vì tổng. Yêu cầu: dùng monotonic stack (cách 3). Lưu ý: một lớp nước rộng `width` cột thì phải cộng `bounded_height` vào **từng** cột nằm giữa hai tường.

```python
water_per_column_stack([0,1,0,2,1,0,1,3,2,1,2,1]) -> [0,0,1,0,1,2,1,0,0,1,0,0]
```

In [26]:
def exercise_water_per_column_stack(height: List[int]) -> List[int]:
    water = [0] * len(height)
    stack = []
    # TODO: giống trap_monotonic_stack, nhưng thay vì total += width * bh,
    #       cộng bh vào water[c] với mọi c nằm giữa left và i
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_water_per_column_stack([0,1,0,2,1,0,1,3,2,1,2,1]) == [0,0,1,0,1,2,1,0,0,1,0,0]
# assert exercise_water_per_column_stack([4,2,0,3,2,5]) == [0,2,4,1,2,0]

### Bài 6 — Mini challenge: LeetCode 11 — Container With Most Water

Cho `height`, chọn **hai đường thẳng** `i < j` sao cho bình chứa tạo bởi chúng (và trục hoành) chứa nhiều nước nhất. Diện tích = `(j - i) * min(height[i], height[j])`. Các cột ở giữa **không** chắn nước (khác bài 42!).

Yêu cầu: `O(n)` bằng two pointers. Câu hỏi tư duy: mỗi bước nên dịch con trỏ ở phía **thấp hơn** hay **cao hơn**? Vì sao?

```python
max_area([1,8,6,2,5,4,8,3,7]) -> 49
max_area([1,1]) -> 1
```

In [27]:
def exercise_max_area(height: List[int]) -> int:
    # TODO: l, r ở hai đầu; tính diện tích; dịch con trỏ phía thấp hơn
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_max_area([1,8,6,2,5,4,8,3,7]) == 49
# assert exercise_max_area([1,1]) == 1

## 14. Gợi ý theo mức

**Bài 1**
- Mức 1: làm đúng như bảng trace cách 1 ở mục 4.
- Mức 2: `maxL` đi từ trái chỉ tăng không giảm; `maxR` đi từ phải cũng vậy.
- Mức 3: (a) `maxL = [3,3,3,3,4]`, `maxR = [4,4,4,4,4]` — tự trừ tiếp.

**Bài 2**
- Mức 1: nhìn `right_max` ngay từ bước 1 là bao nhiêu?
- Mức 2: cột cuối cao 5 là cột cao nhất mảng. Có khi nào `left_max >= 5` trước khi hai con trỏ gặp nhau?
- Mức 3: con trỏ trái đi lần lượt qua cột 0, 1, 2, 3, 4.

**Bài 3**
- Mức 1: chép lại công thức running max ở mục 5.
- Mức 2: vòng lùi viết `for i in range(n - 2, -1, -1):`.
- Mức 3: `left_max[i] = max(left_max[i - 1], height[i])`, `right_max[i] = max(right_max[i + 1], height[i])`.

**Bài 4**
- Mức 1: nếu `height[l] < height[r]` thì `right_max >= height[r] > height[l]`. Còn `left_max` thì sao?
- Mức 2: `left_max` là chiều cao của một cột `k` nào đó mà con trỏ trái **đã rời đi**. Lúc rời `k`, điều kiện `height[k] < height[r']` đúng với một cột `r'` bên phải đã thấy → `right_max > left_max`.
- Mức 3: thân vòng lặp: `if height[l] < height[r]: if height[l] >= left_max: left_max = height[l] else: total += left_max - height[l]; l += 1`.

**Bài 5**
- Mức 1: mỗi lớp nước phủ các cột `left + 1, ..., i - 1`.
- Mức 2: `for c in range(left + 1, i): water[c] += bh`.
- Mức 3: kiểm tra lại: `sum(kết quả)` phải bằng `trap_monotonic_stack(height)`.

**Bài 6**
- Mức 1: diện tích bị giới hạn bởi đường **thấp hơn**.
- Mức 2: nếu dịch phía cao hơn, chiều rộng giảm mà chiều cao vẫn bị chặn bởi phía thấp → không thể tốt hơn.
- Mức 3: `best = max(best, (r - l) * min(height[l], height[r]))`, rồi dịch con trỏ có `height` nhỏ hơn.

## 15. Đáp án tham khảo

Hãy tự làm trước rồi mới xem phần này.

**Bài 1**

| mảng | maxL | maxR | water | tổng |
|---|---|---|---|---:|
| (a) `[3,0,2,0,4]` | `[3,3,3,3,4]` | `[4,4,4,4,4]` | `[0,3,1,3,0]` | **7** |
| (b) `[2,0,2]` | `[2,2,2]` | `[2,2,2]` | `[0,2,0]` | **2** |
| (c) `[5,4,3,2,1]` | `[5,5,5,5,5]` | `[5,4,3,2,1]` | `[0,0,0,0,0]` | **0** |

Mảng (c) giảm dần: bên phải luôn thấp hơn, nước tràn hết ra phải.

**Bài 2**

Ngay bước 1: `left_max = 4`, `right_max = 5` → `4 < 5`. Vì cột 5 (cao 5) là cột cao nhất và `left_max` tối đa chỉ là 4, nên **mọi bước đều dịch con trỏ trái**; con trỏ phải đứng yên ở cột 5.

| bước | cột xử lý | left_max | + nước | total |
|---:|---|---:|---:|---:|
| 1 | 0 | 4 | 0 | 0 |
| 2 | 1 | 4 | 2 | 2 |
| 3 | 2 | 4 | 4 | 6 |
| 4 | 3 | 4 | 1 | 7 |
| 5 | 4 | 4 | 2 | 9 |

Tổng **9**.

**Bài 4 — giải thích bằng lời:** khi `height[l] < height[r]` thì `right_max >= height[r] > height[l]`. Còn `left_max` (nếu lớn hơn `height[l]`) là chiều cao của một cột `k < l` mà con trỏ trái đã rời đi; lúc rời `k` ta có `height[k] < height[r']` với một cột `r'` phía phải, nên `right_max > left_max`. Vậy bên trái đúng là phía thấp hơn, mực nước tại `l` là `max(left_max, height[l])`.

Cell dưới kiểm chứng bài 1, bài 2 bằng máy, sau đó là đáp án code bài 3–6.

In [28]:
# Kiểm chứng Bài 1 và Bài 2
assert trap_two_pointers([3, 0, 2, 0, 4]) == 7
assert trap_two_pointers([2, 0, 2]) == 2
assert trap_two_pointers([5, 4, 3, 2, 1]) == 0
draw_elevation([3, 0, 2, 0, 4], [0, 3, 1, 3, 0])
print()
_ = trace_two_pointers([4, 2, 0, 3, 2, 5])

4 |    #|
3 |#~~~#|
2 |#~#~#|
1 |#~#~#|
  +-----+
   01234

bước |  l |  r | lmax | rmax | xử lý      | +  | total
   1 |  0 |  5 |    4 |    5 | cột  0 trái |  0 |     0
   2 |  1 |  5 |    4 |    5 | cột  1 trái |  2 |     2
   3 |  2 |  5 |    4 |    5 | cột  2 trái |  4 |     6
   4 |  3 |  5 |    4 |    5 | cột  3 trái |  1 |     7
   5 |  4 |  5 |    4 |    5 | cột  4 trái |  2 |     9
dừng: l = r = 5, height[5] = 5 (max của mảng = 5)


In [29]:
def answer_trap_prefix_suffix(height: List[int]) -> int:
    n = len(height)
    left_max = [0] * n
    right_max = [0] * n
    left_max[0] = height[0]
    for i in range(1, n):
        left_max[i] = max(left_max[i - 1], height[i])
    right_max[n - 1] = height[n - 1]
    for i in range(n - 2, -1, -1):
        right_max[i] = max(right_max[i + 1], height[i])
    total = 0
    for i in range(n):
        total += min(left_max[i], right_max[i]) - height[i]
    return total


def answer_trap_two_pointers_v2(height: List[int]) -> int:
    l, r = 0, len(height) - 1
    left_max = right_max = 0
    total = 0
    while l < r:
        if height[l] < height[r]:
            if height[l] >= left_max:
                left_max = height[l]
            else:
                total += left_max - height[l]
            l += 1
        else:
            if height[r] >= right_max:
                right_max = height[r]
            else:
                total += right_max - height[r]
            r -= 1
    return total


def answer_water_per_column_stack(height: List[int]) -> List[int]:
    water = [0] * len(height)
    stack = []
    for i, h in enumerate(height):
        while stack and h > height[stack[-1]]:
            bottom = stack.pop()
            if not stack:
                break
            left = stack[-1]
            bh = min(height[left], h) - height[bottom]
            for c in range(left + 1, i):
                water[c] += bh
        stack.append(i)
    return water


def answer_max_area(height: List[int]) -> int:
    l, r = 0, len(height) - 1
    best = 0
    while l < r:
        best = max(best, (r - l) * min(height[l], height[r]))
        if height[l] < height[r]:
            l += 1
        else:
            r -= 1
    return best


assert answer_trap_prefix_suffix([0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]) == 6
assert answer_trap_prefix_suffix([4, 2, 0, 3, 2, 5]) == 9
assert answer_trap_prefix_suffix([7]) == 0
assert answer_trap_two_pointers_v2([0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]) == 6
assert answer_trap_two_pointers_v2([4, 2, 0, 3, 2, 5]) == 9
assert answer_trap_two_pointers_v2([5, 0, 0, 0, 1]) == 3
assert answer_water_per_column_stack([0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]) == [0, 0, 1, 0, 1, 2, 1, 0, 0, 1, 0, 0]
assert answer_water_per_column_stack([4, 2, 0, 3, 2, 5]) == [0, 2, 4, 1, 2, 0]
assert answer_max_area([1, 8, 6, 2, 5, 4, 8, 3, 7]) == 49
assert answer_max_area([1, 1]) == 1

# Đối chiếu ngẫu nhiên các đáp án với brute force
random.seed(7)
for _ in range(500):
    h = [random.randint(0, 6) for _ in range(random.randint(1, 10))]
    assert answer_trap_prefix_suffix(h) == trap_bruteforce(h)
    assert answer_trap_two_pointers_v2(h) == trap_bruteforce(h)
    assert sum(answer_water_per_column_stack(h)) == trap_bruteforce(h)
print("Đáp án tham khảo chạy PASS toàn bộ test!")

Đáp án tham khảo chạy PASS toàn bộ test!


## 16. Template ngắn nên nhớ

```python
class Solution:
    def trap(self, height: List[int]) -> int:
        l, r = 0, len(height) - 1
        left_max = right_max = 0
        total = 0
        while l < r:
            left_max = max(left_max, height[l])
            right_max = max(right_max, height[r])
            if left_max < right_max:        # bên trái là "thanh gỗ ngắn"
                total += left_max - height[l]
                l += 1
            else:                           # bên phải là "thanh gỗ ngắn"
                total += right_max - height[r]
                r -= 1
        return total
```

Câu thần chú: **"Nước tại cột = tường thấp hơn trong hai tường cao nhất − chiều cao cột. Bên nào thấp hơn thì bên đó đã biết đủ, xử lý bên đó."**

## 17. Checklist trước khi submit

- Mình có **cập nhật max trước** khi cộng nước (để không bao giờ cộng số âm) chưa?
- Mình có xử lý cột ở phía có max **thấp hơn** (không phải phía cao hơn) chưa?
- Điều kiện vòng lặp là `l < r` (cột gặp nhau là cột cao nhất, không chứa nước)?
- Hàm trả về **một số nguyên** tổng nước, không phải list?
- Đã thử `n = 1`, mảng tăng dần, giảm dần, toàn bằng nhau, và hai example của đề?
- Nếu dùng monotonic stack: có `if not stack: break` và `width = i - left - 1` chưa?